# Lab 12: How to answer from documents with retrieval

**Notebook 2 of 2: retrieval at the Chinook support desk**

## Goal

Answer a support rep's questions from the Chinook music shop's own files: 20 policies (the shop's rules) and 16
support tickets (cases where a rule was applied). We store all 36 files in Chroma, search them by meaning, and let
the model answer only from what the search found. Then a rep asks *When is support staffed?* The nearest file is the
one about the support staff, not the one with the hours. With only the nearest file in the prompt, the model replies
with a careful "I don't know", and no error tells us the search missed. Last, we narrow a search to the policies
with a filter, and keep the collection on disk so the next program can open it without embedding anything again.

## What you will learn

- How to store the shop's files in a Chroma collection with ids, texts, labels and vectors, and look up a record by
  its id.
- How to answer a rep's question only from the retrieved files, and read which files were used.
- How to spot a search that put the wrong file first, when the reply raises no error.
- How to narrow a search with a metadata filter, and what a filter cannot do.
- How to keep a collection on disk with `PersistentClient` and open it again from the folder.

## From the house notes to the shop

Notebook 1 built retrieval on seven notes. It is five steps:

1. Once, embed every document: one vector of numbers per text, from the embedding model.
2. Store the vectors with their names and texts in a vector store.
3. For each question, embed the question with the same model.
4. Search the store for the `k` nearest documents (the highest cosine similarity: how closely two vectors point the
   same way, 1 for the same direction and 0 for nothing in common).
5. Pack those documents with their names into the prompt, and ask the model to answer only from them, or to say it
   does not know.

Now we take it seriously. A rep repeats what the model says to a customer, so a missed rule or a wrong answer
matters. This notebook restates everything it uses, so it runs on its own.

## Setup

The first cell is the same in every lab: it finds the repository folder, reads the model names from `models.env`, and
takes the key from `.env` without printing it.

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


One client, with a 60-second limit on each request, and the embedding model from `models.env`. Every chat call
passes `reasoning_effort=EFFORT or omit` (`omit` leaves the argument out when `models.env` sets no effort).

In [2]:
from openai import OpenAI, omit

client = OpenAI(timeout=60)
EMBEDDING_MODEL = settings["EMBEDDING_MODEL"]

---

## Part A: Store the shop's files in a Chroma collection

The files are Markdown, in `data/corpus/` at the repository folder. We read them in place and never change them.

In [3]:
CORPUS = ROOT / "data" / "corpus"
files = sorted(CORPUS.glob("*.md"))
names = [p.name for p in files]
texts = [p.read_text() for p in files]

print(len(files), "files:",
      sum(name.startswith("policy_") for name in names), "policies,",
      sum(name.startswith("ticket_") for name in names), "tickets")
print(names[:3], "...")

36 files: 20 policies, 16 tickets
['policy_accounts.md', 'policy_cancellations.md', 'policy_damaged_media.md'] ...


`names` and `texts` come from the same sorted list of files, so position `i` in one belongs to position `i` in the
other. Here is one policy, as a rep would read it:

In [4]:
print(texts[names.index("policy_returns.md")])

# Returns

Physical items (CDs, vinyl, boxed sets) may be returned within 30 days of the invoice date if unopened.

Opened physical media can be returned within 14 days only if the disc is defective. A replacement is preferred over a refund.

Digital downloads and streamed albums cannot be returned. See policy_digital_downloads.md.

Start a return by writing to support. Include the invoice number. Support will reply within the SLA in policy_support_hours.md.



Four rules in one short file: 30 days for an unopened physical item, 14 days for an opened one only if it is
defective, no returns on digital albums, and how to start a return. Every file is short and keeps to one topic, so
one file will be one record in the store. (A long document would first be split into smaller pieces, called
**chunks**; the extra notebook of this lab does that.)

### Embed the 36 files in one call

`input` takes a list of texts and returns one vector per text, in the same order.

In [5]:
response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
vectors = [item.embedding for item in response.data]

print(len(vectors), "vectors of", len(vectors[0]), "numbers")
print("tokens read:", response.usage.prompt_tokens)

36 vectors of 1536 numbers
tokens read: 2551


36 vectors, one per file, each of 1,536 numbers, from 2,551 tokens read: the whole folder in one call, for about 0.00005 US dollars at the embedding price in `models.env`.

### Add the files to a collection

A Chroma record has four parts, passed as four lists in the same order: an id, a text, a vector, and **metadata**,
labels we choose. `name.split("_")[0]` takes the part of the file name before the first underscore, so every file
is labelled `{"kind": "policy"}` or `{"kind": "ticket"}`. Part D searches on that label.

In [6]:
import chromadb

metadatas = [{"kind": name.split("_")[0]} for name in names]

chroma = chromadb.EphemeralClient()
collection = chroma.get_or_create_collection("shop_files", configuration={"hnsw": {"space": "cosine"}})
collection.add(ids=names, documents=texts, metadatas=metadatas, embeddings=vectors)

print("stored:", collection.count())
print("distance:", collection.configuration["hnsw"]["space"])

stored: 36
distance: cosine


- `EphemeralClient` keeps the collection in this process's memory; Part E keeps one on disk.
- We passed our own vectors, so Chroma embeds nothing: OpenAI embeds, Chroma stores and ranks.
- In the `cosine` space Chroma reports a **distance** of `1 − cosine`: smaller is nearer. We print the cosine,
  `1 - distance`, from here on.

`get` fetches records by id, without any search. Here is what Chroma keeps for one file:

In [7]:
collection.get(ids=["policy_returns.md"], include=["metadatas", "documents"])

{'ids': ['policy_returns.md'],
 'embeddings': None,
 'documents': ['# Returns\n\nPhysical items (CDs, vinyl, boxed sets) may be returned within 30 days of the invoice date if unopened.\n\nOpened physical media can be returned within 14 days only if the disc is defective. A replacement is preferred over a refund.\n\nDigital downloads and streamed albums cannot be returned. See policy_digital_downloads.md.\n\nStart a return by writing to support. Include the invoice number. Support will reply within the SLA in policy_support_hours.md.\n'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'kind': 'policy'}]}

The file name is the id, so every result points back to a file a rep can open. `get` takes no question, so each field is a flat list with one entry
per record; a `query` result, next, has one inner list per question.

### Query the collection, then write a helper for it

A search embeds the question with the same model and asks for the nearest vectors. We print the ids and distances of
the raw result first.

In [8]:
QUESTION = "How long do I have to return an unopened CD?"

q = client.embeddings.create(model=EMBEDDING_MODEL, input=QUESTION).data[0].embedding
got = collection.query(query_embeddings=[q], n_results=3)
print(got["ids"])
print(got["distances"])

[['policy_returns.md', 'policy_warranty.md', 'policy_damaged_media.md']]
[[0.38673484325408936, 0.5359547138214111, 0.5701746940612793]]


Every field is a list of lists: `query_embeddings` takes a list of questions, and the result holds one inner list per
question. We sent one, so `got["ids"][0]` is our list, nearest first. `policy_returns.md` is first, at a distance of 0.3867, which is a cosine of 1 − 0.3867 = 0.6133.

Every search from here on repeats these steps, so two small functions do them. `embed` returns one vector.
`retrieve` returns the same ids, texts and distances as the raw result, one dictionary per file, with the cosine in
place of the distance. `where` is passed to the query unchanged; Part D explains it, and `None` means no filter.

In [9]:
def embed(text):
    return client.embeddings.create(model=EMBEDDING_MODEL, input=text).data[0].embedding


def retrieve(question, k=3, where=None):
    got = collection.query(query_embeddings=[embed(question)], n_results=k, where=where)
    rows = []
    for file_id, text, distance in zip(got["ids"][0], got["documents"][0], got["distances"][0]):
        rows.append({"id": file_id, "text": text, "cosine": round(1 - distance, 4)})
    return rows


for row in retrieve(QUESTION):
    print(row["cosine"], row["id"])

0.6133 policy_returns.md
0.464 policy_warranty.md
0.4299 policy_damaged_media.md


The same three files in the same order. (The fourth decimal of a cosine can move between two runs: the question was
embedded again, and the same text embedded twice can give slightly different numbers.)

**Checkpoint A.** 36 files, 20 policies and 16 tickets; 36 vectors of 1,536 numbers; `stored: 36` in the `cosine` space;
`policy_returns.md` first for the returns question.

---

## Part B: Answer a rep's question from the retrieved files

Retrieval returns files; the rep wants a sentence. We pack the retrieved files under their names and ask the model to
answer only from them, with a rule for when they do not answer.

In [10]:
RULES = ("Answer the question using only the documents below. "
         "If they do not contain the answer, say you do not know.")


def answer(question, k=3, where=None):
    hits = retrieve(question, k, where)
    packed = "\n\n".join(f"[{row['id']}]\n{row['text']}" for row in hits)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": RULES},
            {"role": "user", "content": f"Documents:\n\n{packed}\n\nQuestion: {question}"},
        ],
        max_completion_tokens=200,
        reasoning_effort=EFFORT or omit,
    )
    return [row["id"] for row in hits], response.choices[0].message.content, response.usage.prompt_tokens

`answer` runs three steps: `retrieve`, pack each file under its name in square brackets, and one chat call with
`RULES` as the system message. It returns three things: the files used (the reply alone does not say what the model
was given), the reply, and the prompt tokens of the call. `max_completion_tokens=200` caps the reply's length.

In [11]:
used, reply, tokens = answer(QUESTION)
print("files used:", used)
print(reply)
print("prompt tokens:", tokens)

files used: ['policy_returns.md', 'policy_warranty.md', 'policy_damaged_media.md']
You can return an unopened CD within 30 days of the invoice date.
prompt tokens: 290


In our run: *You can return an unopened CD within 30 days of the invoice date.* The 30-day rule, from the first file. The prompt was 290 tokens: three short files and a question, not the whole folder.

**Checkpoint B.** The returns question gets the 30-day rule from `policy_returns.md`, with three files in a prompt of
290 tokens.

---

## Part C: Find the miss when the nearest file is not the one that answers

*When is support staffed?* The answer is the first line under the heading of `policy_support_hours.md`: Monday to Friday, 09:00 to
17:00. Look at the search before any answer:

In [12]:
STAFFED = "When is support staffed?"

for row in retrieve(STAFFED):
    print(row["cosine"], row["id"])

0.5276 policy_support_reps.md
0.4725 policy_support_hours.md
0.3893 policy_escalations.md


`policy_support_reps.md` comes first (cosine 0.5276), and the file with the hours second (0.4725). The representatives file is about the support staff (who answers tickets, who is on duty), so its meaning
is close to the question; it just does not say *when*.

Before you run it: with `k=1`, only that file goes to the model. What will the rep read?

In [13]:
used, reply, tokens = answer(STAFFED, k=1)
print("files used:", used)
print(reply)

files used: ['policy_support_reps.md']
I don’t know. The document names the support representatives and describes how tickets are handled, but it doesn’t say when support is staffed.


In our run: *I don’t know. The document names the support representatives and describes how tickets are handled, but it doesn’t say when support is staffed.*

The model did its job: it answered honestly from the one file it was given, and that file does not say. For the rep
the answer is still wrong, because the folder does say. The run raised no error and the reply sounds careful. **Only
the list of files used shows that the search missed.** A rep who sees only the reply would tell the customer the shop
has no hours on record.

With `k=3` the hours file is in the prompt:

In [14]:
used, reply, tokens = answer(STAFFED, k=3)
print("files used:", used)
print(reply)
print("prompt tokens:", tokens)

files used: ['policy_support_reps.md', 'policy_support_hours.md', 'policy_escalations.md']
Support is staffed Monday to Friday, 09:00–17:00 shop local time, excluding public holidays.
prompt tokens: 296


In our run: *Support is staffed Monday to Friday, 09:00–17:00 shop local time, excluding public holidays.* That is the first line under the heading of the second file, from a prompt of 296 tokens.

A bigger `k` fixed this question, and every prompt now carries three files, most of them about something else. How
often the right file is first, or in the top three, over many questions, is measured in the extra notebook.

**In production:** keep the ids of the retrieved files with every answer (in a log, or as sources shown to the rep),
so a reply like the `k=1` one can be traced to a search that missed.

**Checkpoint C.** `policy_support_reps.md` first, `policy_support_hours.md` second; with `k=1` the reply says the
document does not say when support is staffed; with `k=3` it gives Monday to Friday, 09:00 to 17:00.

---

## Part D: Narrow the search with a metadata filter

The folder mixes rules (policies) with cases (tickets). A ticket can tell one customer's story in words close to a
question and rank above the rule itself. A **metadata filter** limits the search to the records whose labels match,
before the ranking. `where={"kind": "policy"}` keeps only the records labelled `policy`.

In [15]:
DOWNLOAD = "My download link expired. Can I get a new one?"

for where in [None, {"kind": "policy"}]:
    print("where =", where)
    for row in retrieve(DOWNLOAD, where=where):
        print("   ", row["cosine"], row["id"])

where = None


    0.5452 ticket_02_puja_download_expired.md
    0.5044 policy_digital_downloads.md
    0.2544 policy_damaged_media.md
where = {'kind': 'policy'}


    0.5044 policy_digital_downloads.md
    0.2544 policy_damaged_media.md
    0.2428 policy_returns.md


Without the filter, Puja's ticket, `ticket_02_puja_download_expired.md`, comes first (0.5452), and the policy second. The ticket is about one customer, with her dates and her rep. A rep answering another customer needs the rule, and with `k=1` only the ticket would reach the model.

With `where={"kind": "policy"}` the tickets are gone and `policy_digital_downloads.md` is first, with the same cosine as before (0.5044; on your run the fourth decimal may differ, because each call embeds the question again). **The filter removes the tickets before the ranking, and the policies that stay keep the same cosines.**

**What a filter cannot do:** it narrows the search to what you allowed, including when you allowed the wrong thing.
This question is answered by a ticket alone (ticket 06: the replacement shipped on 2012-07-20):

In [16]:
SHIPPED = "When was the replacement for the warped vinyl shipped?"

for where in [None, {"kind": "policy"}]:
    print("where =", where, "->", [row["id"] for row in retrieve(SHIPPED, where=where)])

where = None -> ['ticket_06_germany_warped_vinyl.md', 'policy_damaged_media.md', 'ticket_01_helena_wrong_album.md']


where = {'kind': 'policy'} -> ['policy_damaged_media.md', 'policy_warranty.md', 'policy_returns.md']


Without the filter, ticket 06 is first. With the policies-only filter it cannot be found at all, and the model would get three policies, none of which holds the date.

**In production:** metadata filters carry business rules into the search: a product line, a date range, a language,
and above all permissions (a rep searches only the tickets of their own customers).

**Checkpoint D.** Without the filter Puja's ticket comes first for the download question; with it no ticket appears,
and `policy_digital_downloads.md` is first with the same cosine. For the shipping date, the policies-only filter
leaves ticket 06 out.

---

## Part E: Keep the collection on disk with `PersistentClient`

Everything so far lives in this notebook's memory: restart the kernel and the collection is gone, and the 36 files
would have to be embedded again. `PersistentClient` keeps a collection in a folder on disk. The collection API is the
same; only the client changes.

The folder is `chroma_db/shop` inside this lab's folder. The repository ignores every `chroma_db/` folder, so nothing
written there is committed.

In [17]:
STORE = Path("chroma_db") / "shop"

disk = chromadb.PersistentClient(path=str(STORE))
shop = disk.get_or_create_collection("shop_files", configuration={"hnsw": {"space": "cosine"}})
shop.upsert(ids=names, documents=texts, metadatas=metadatas, embeddings=vectors)

print("stored on disk:", shop.count())
print("folder:", STORE, "holds", sorted(path.name for path in STORE.iterdir()))

stored on disk: 36
folder: chroma_db/shop holds ['chroma.sqlite3', 'd611c9f6-d5b5-4ac7-aedc-c35d8cc6e3ea']


- `upsert` adds the records that are new and replaces the ones whose id already exists, so running the cell again
  does not try to add the 36 files twice.
- The folder holds `chroma.sqlite3`, a SQLite database with the records, and a folder with an id Chroma generated
  for the search index (not the collection's id).

The next cell opens the folder with a new client and uses nothing from above, so it also works after a kernel
restart. Try it: restart the kernel and run only that cell.

In that cell, `splitlines()[2]` is the file's third line: the heading, a blank line, then the hours.

In [18]:
from pathlib import Path
import chromadb

reopened = chromadb.PersistentClient(path=str(Path("chroma_db") / "shop")).get_collection("shop_files")
print("read from disk:", reopened.count())

record = reopened.get(ids=["policy_support_hours.md"], include=["documents"])
print(record["documents"][0].splitlines()[2])

read from disk: 36
Support is staffed Monday to Friday, 09:00 to 17:00 in the shop's local time. Closed public holidays.


36 records, and the hours line of `policy_support_hours.md`: *Support is staffed Monday to Friday, 09:00 to 17:00 in the shop's local time. Closed public holidays.* No embedding call was made. The texts, labels and vectors are all in the folder: add `"embeddings"` to `include` to see the vectors, and a `query` on `reopened` would need only the question's embedding.

Chroma has three clients, and the collection API is the same for all of them:

| Client | Where the data lives | Who can search it | Use it for |
|---|---|---|---|
| `EphemeralClient()` | this process's memory | this notebook | tests and small collections rebuilt on every run |
| `PersistentClient(path=...)` | a folder on disk | one program at a time | one program whose data must survive a restart |
| `HttpClient(host=..., port=...)` | the folder of a Chroma server | any program that reaches the server | a store shared by several programs (the extra notebook) |

**In production:** the same pattern with a managed or self-hosted database: Pinecone, Qdrant, pgvector in
PostgreSQL, or Chroma's own hosted service.

**Checkpoint E.** `stored on disk: 36`, a folder with `chroma.sqlite3` in it, then `read from disk: 36` and the
hours line of `policy_support_hours.md` from a new client.

---

## What you can do now

- Store the shop's files in a Chroma collection with ids, texts, `kind` labels and our own vectors, and fetch one
  record with `get` (Part A).
- Answer a rep's question only from the retrieved files, and print the files used beside the reply (Part B).
- Spot a search that put the wrong file first: a careful "I don't know" with no error, and the files used (Part C).
- Narrow a search with `where`, and say why a filter changes which files are ranked but not their scores (Part D).
- Keep a collection on disk with `PersistentClient` and open it again from a new client (Part E).

## Troubleshooting

| What you see | Why | Fix |
|---|---|---|
| `KeyError: 'EMBEDDING_MODEL'` | `models.env` has no embedding model | add an `EMBEDDING_MODEL` line to `models.env` (the repository's copy has one) and run the first two cells again |
| `NotFoundError: Collection [shop_files] does not exist` in the last cell | the folder `chroma_db/shop` was never written, or was deleted | run the cell that writes it (Part E's first cell) |
| your cosines differ from ours in the third or fourth decimal | the same text embedded twice can give slightly different numbers | check the order of the files, not the last digits |

## Challenge

1. **Choose the filter that can answer.** Answer *When was the replacement for the warped vinyl shipped?* with `k=1`
   twice: once with `where={"kind": "ticket"}` and once with `where={"kind": "policy"}`. Print the files used and
   the reply each time, and say which filter was the right choice for this question.
2. **Make the reply cite its files, and check the citations in code.** Write new rules that ask the reply to end
   with a line `Sources:` followed by the file names it used. Then write a check that reads that line and confirms
   every cited name is among the retrieved files. Run it on the returns question and on *When is support staffed?*
   with `k=1`.

**Expected outcome.** 1. With the tickets-only filter the search uses `ticket_06_germany_warped_vinyl.md` and the reply gives the date, 2012-07-20; with the policies-only filter it uses `policy_damaged_media.md` and the reply says it does not know when the replacement was shipped. 2. Both replies end with a `Sources:` line: the returns question cites `policy_returns.md`, the staffing question at `k=1` cites `policy_support_reps.md`, and the check passes on both, also on the staffing reply that does not answer the question. Your replies' wording will differ. A worked solution is in `solved/12b_retrieval_at_the_desk_solution.ipynb`.

## Resources

- [Chroma: adding data](https://docs.trychroma.com/docs/collections/add-data) and
  [updating data](https://docs.trychroma.com/docs/collections/update-data): `add` and `upsert`.
- [Chroma: query and get](https://docs.trychroma.com/docs/querying-collections/query-and-get): `query`, `get`,
  `include`, and the shape of the result.
- [Chroma: metadata filtering](https://docs.trychroma.com/docs/querying-collections/metadata-filtering): `where` and
  its operators.
- [Chroma: clients](https://docs.trychroma.com/docs/run-chroma/clients): `PersistentClient` and its folder, and the
  other clients.
- [OpenAI: vector embeddings](https://developers.openai.com/api/docs/guides/embeddings): the embedding models and
  their sizes.
- [Bonus lab B6: How to move retrieval to Pinecone](../bonus/B6_Pinecone_Retrieval/README.md): the same search on Pinecone,
  a managed vector database (it needs a Pinecone account).